# LLM Tokenization and Embedding

In [1]:
MODEL_ID = "openai-community/gpt2"

## Tokenization

In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

prompt = "The theory of relativity is cool."
token_ids = tokenizer.encode(prompt)
print(token_ids)

[464, 4583, 286, 44449, 318, 3608, 13]


In [3]:
from transformers import AutoModel
model = AutoModel.from_pretrained(MODEL_ID)

In [4]:
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [5]:
# we need the token as pytorch tensors
input_ids = tokenizer.encode(prompt, return_tensors="pt")
input_ids

tensor([[  464,  4583,   286, 44449,   318,  3608,    13]])

## Embedding

In [6]:
print("embed dim :", model.embed_dim)
print("vocab size:", tokenizer.vocab_size)

embed dim : 768
vocab size: 50257


In [7]:
WE = model.wte.weight  # Word Token Embeddings 
PE = model.wpe.weight  # Word Position Embeddings

print(WE.shape)
print(PE.shape) # context window length == 1024 tokens

torch.Size([50257, 768])
torch.Size([1024, 768])


In [8]:
input_ids = tokenizer.encode("The")  #return_tensors="pt")
input_ids

[464]

In [9]:
# onehot
import torch
import numpy as np
one_hot = np.zeros(tokenizer.vocab_size) 
one_hot[input_ids[0]-1] = 1
one_hot = torch.tensor(one_hot, dtype=torch.float32) # [0, 0, 0, 0, ..., 1, 0, ..., 0] # item number 464 is 1.

# embedding
embedding = torch.matmul(WE.T,one_hot)
print(embedding.shape)
print(embedding)


torch.Size([768])
tensor([-1.9558e-01,  1.7521e-01,  1.0660e-01, -1.3306e-01,  2.1067e-02,
         2.0831e-01, -3.1553e-01, -6.4046e-02,  1.9117e-02, -1.3318e-01,
         5.5814e-02,  1.8347e-02,  3.3747e-02, -1.4204e-01, -4.5536e-02,
        -7.3230e-02,  6.6782e-02, -4.0008e-02,  1.9214e-01,  2.8787e-01,
         2.2954e-01, -1.3426e-01,  1.7875e-01,  1.4024e-01,  2.3283e-01,
        -1.7405e-01,  1.4877e-01, -2.2481e-01,  2.0865e-01,  1.1210e-01,
         1.3062e-01,  1.6762e-01, -2.3032e-01,  1.8145e-01,  3.3456e-01,
        -1.4299e-02, -3.1738e-01, -4.6511e-02,  2.2055e-01, -2.2793e-01,
        -8.2367e-02,  1.1124e-01, -1.7095e-01, -2.8625e-02, -6.8895e-02,
        -1.7352e-01, -2.5119e-02, -4.8197e-02, -2.4616e-01, -1.8915e-01,
        -1.3774e-01,  5.4413e-02,  1.6038e-01,  1.5343e-01, -9.6327e-02,
        -2.8944e-01,  9.6194e-02,  1.1039e-01,  6.1438e-02, -7.9496e-02,
         2.2518e-02, -1.1462e-01, -1.0346e-01,  1.9272e-01,  8.1632e-02,
        -1.7026e-02, -1.4185e-01,

In [10]:
### Context Window Size
print(model.config.n_ctx)


1024


In [11]:
######### positional embedding ##########
# onehot
import torch
import numpy as np
one_hot = np.zeros(model.config.n_ctx) 
one_hot[0] = len(input_ids)
one_hot = torch.tensor(one_hot, dtype=torch.float32) # [1, 0, 0, 0, ..., 0] 

# positional embedding
pos_embedding = torch.matmul(PE.T,one_hot)
print(pos_embedding.shape)
print(pos_embedding)

torch.Size([768])
tensor([-1.8821e-02, -1.9742e-01,  4.0267e-03,  1.1347e-02,  6.3824e-02,
        -1.0501e-01,  3.6937e-02, -1.6803e-01, -4.9111e-02, -5.6461e-02,
        -2.4560e-03,  1.3503e-02, -4.1711e-03,  1.5115e-02,  1.6595e-02,
        -1.3808e-01, -6.3314e-03, -4.6150e-02,  2.6675e-02, -2.0417e-01,
         1.3454e-02, -3.6267e-02,  1.9301e-02, -2.5931e-02,  8.0243e-03,
         8.4712e-03, -1.9906e-02,  6.6802e-02,  7.1151e-03, -2.6618e-02,
         2.0829e-02, -3.3732e-02, -8.2898e-03,  9.8622e-03, -2.7369e-02,
        -9.9118e-02, -7.5254e-01,  2.3550e-02, -3.0513e-02,  7.7456e-02,
         3.4301e-03,  7.1132e-03,  2.6479e-02, -1.2113e-03,  1.1219e-01,
        -2.0606e-03, -2.2458e-02, -2.2287e-02,  2.3570e-02,  3.9777e-01,
         1.8856e-02,  2.0280e-02,  6.3043e-01,  2.3146e-02, -4.6894e-02,
         4.0653e+00, -1.7403e-02, -5.1683e-02,  7.2271e-02, -7.9312e-02,
         4.0248e-02,  1.9908e-02, -4.6380e-02, -2.8380e-02,  7.2535e-03,
         2.6772e-02,  1.4972e-03,